[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/39_ppo_loss.ipynb)

# 🔴 Hard: PPO Clipped Loss

Implement the **PPO (Proximal Policy Optimization)** **clipped surrogate loss**.

Given:
- `new_logps`: current policy log-probs $(B,)$
- `old_logps`: old policy log-probs $(B,)$
- `advantages`: advantage estimates $(B,)$

Define the ratio

$$ r_i = \exp(\text{new\_logps}_i - \text{old\_logps}_i). $$

Then compute
- $L^{\text{unclipped}}_i = r_i A_i$
- $L^{\text{clipped}}_i = \operatorname{clip}(r_i, 1-\epsilon, 1+\epsilon) A_i$

The loss is the negative batch mean of the elementwise minimum:

$$
\mathcal{L}_\text{PPO} = -\mathbb{E}_i\big[\min(L^{\text{unclipped}}_i, L^{\text{clipped}}_i)\big].
$$

Implementation notes: detach `old_logps` and `advantages` so gradients only flow through `new_logps`.

### Signature
```python
from torch import Tensor

def ppo_loss(new_logps: Tensor, old_logps: Tensor, advantages: Tensor,
             clip_ratio: float = 0.2) -> Tensor:
    """PPO clipped surrogate loss over a batch."""
```


In [2]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.8 MB/s eta 0:00:00


In [3]:
import torch
import torch.nn.functional as F
from torch import Tensor


In [25]:
# ✏️ YOUR IMPLEMENTATION HERE

def ppo_loss(new_logps: Tensor, old_logps: Tensor, advantages: Tensor,
             clip_ratio: float = 0.2) -> Tensor:
    ratio = torch.exp(new_logps - old_logps.detach())
    l_clipped = ratio.clamp(1-clip_ratio, 1+clip_ratio)*advantages.detach()
    l_unclipped = ratio*advantages.detach()
    return -torch.minimum(l_clipped, l_unclipped).mean()
    pass  # -mean(min(r * adv, clamp(r, 1-clip, 1+clip) * adv)) with gradients only through new_logps


In [26]:
# 🧪 Debug
new_logps = torch.tensor([0.0, -0.2, -0.4, -0.6])
old_logps = torch.tensor([0.0, -0.1, -0.5, -0.5])
advantages = torch.tensor([1.0, -1.0, 0.5, -0.5])
print('Loss:', ppo_loss(new_logps, old_logps, advantages, clip_ratio=0.2))


Loss: tensor(-0.0488)


In [27]:
# ✅ SUBMIT
from torch_judge import check
check('ppo_loss')



🧪 Testing: PPO (Proximal Policy Optimization) Clipped Loss (Hard)
──────────────────────────────────────────────────
  ✅ [1/3] Basic shape & type (2.6ms)
  ✅ [2/3] Numeric check vs fixed value (0.5ms)
  ✅ [3/3] Gradient flows to new_logps only (0.6ms)
──────────────────────────────────────────────────
  🎉 All 3 tests passed! (3.7ms total)
  Progress saved. Run status() to see your dashboard.

